<a href="https://colab.research.google.com/github/asaveraasad-data/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## SETUP:

In [1]:
!pip -q install duckdb datasets huggingface_hub pyarrow

In [2]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

print("✅ Connected to Hugging Face")

✅ Connected to Hugging Face


In [3]:
from datasets import load_dataset

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_clients",
    split="train"
)

print(ds)

Dataset({
    features: ['client_hash_id', 'is_active', 'has_gsc_access', 'has_ga4_access', 'access_profile', 'client_created_date', 'client_updated_date', 'gsc_data_start', 'ga4_data_start'],
    num_rows: 104
})


In [4]:
from datasets import get_dataset_config_names

configs = get_dataset_config_names("FlyRank/internship-warehouse")

print(configs)

['dim_clients', 'dim_content', 'fact_content_daily_performance', 'fact_content_query_90d']


In [5]:
from datasets import load_dataset

daily = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True,
)

first = next(iter(daily))

print(first.keys())

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

dict_keys(['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'])


In [6]:
first

{'report_date': datetime.date(2025, 1, 27),
 'client_hash_id': 'client_9958f0a7ae1df715',
 'content_hash_id': 'content_3b70a18ea133b2bb',
 'client_has_gsc': True,
 'client_has_ga4': True,
 'gsc_data_available': True,
 'ga4_data_available': False,
 'gsc_impressions': 30,
 'gsc_clicks': 0,
 'gsc_sum_position': 115,
 'gsc_avg_position': 3.8333333333333335,
 'ga4_pageviews': 0,
 'ga4_sessions': 0,
 'ga4_users': 0,
 'ga4_engaged_sessions': 0,
 'ga4_total_engagement_sec': 0,
 'sessions_organic': 0,
 'sessions_direct': 0,
 'sessions_referral': 0,
 'sessions_social': 0,
 'sessions_paid': 0,
 'sessions_ai': 0,
 'ai_chatgpt': 0,
 'ai_perplexity': 0,
 'ai_gemini': 0,
 'ai_copilot': 0,
 'ai_claude': 0,
 'ai_meta': 0,
 'ai_other': 0,
 'scroll_events': 0}

In [7]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)

# Show only performance table files
for f in files:
    if "fact_content_daily_performance" in f:
        print(f)

fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/month=2026-04/data_0.parquet
fact_content_daily_performance/month=202

In [8]:
from huggingface_hub import hf_hub_download

march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet"
)

print(march_file)

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [9]:
import duckdb

con = duckdb.connect()

df = con.sql(f"""
SELECT *
FROM read_parquet('{march_file}')
""").df()

print(df.head())
print(df.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  report_date           client_hash_id           content_hash_id  \
0  2026-03-01  client_73cda7b4e4f265ea  content_b7e512995f79d5a6   
1  2026-03-01  client_73cda7b4e4f265ea  content_05597932fe4da067   
2  2026-03-01  client_73cda7b4e4f265ea  content_7a105f548d9c6916   
3  2026-03-01  client_73cda7b4e4f265ea  content_905aa32a0230694e   
4  2026-03-01  client_73cda7b4e4f265ea  content_a3ea9792f793ec72   

   client_has_gsc  client_has_ga4  gsc_data_available  ga4_data_available  \
0            True           False                True                <NA>   
1            True           False                True                <NA>   
2            True           False                True                <NA>   
3            True           False                True                <NA>   
4            True           False                True                <NA>   

   gsc_impressions  gsc_clicks  gsc_sum_position  ...  sessions_ai  \
0               20           0                67  ...     

In [10]:
duplicates = (
    df.groupby(["report_date", "client_hash_id", "content_hash_id"])
      .size()
      .reset_index(name="count")
)

duplicates[duplicates["count"] > 1]

,report_date,client_hash_id,content_hash_id,count


In [11]:
print("Rows:", len(df))
print("Start:", df["report_date"].min())
print("End:", df["report_date"].max())

Rows: 9841378
Start: 2026-03-01 00:00:00
End: 2026-03-31 00:00:00


In [12]:
available = df[df["gsc_data_available"] == True]

print("Rows with GSC available:", len(available))
print("Percentage:", round(len(available) / len(df) * 100, 2), "%")

Rows with GSC available: 3611061
Percentage: 36.69 %


In [13]:
import os
import subprocess

REPO_URL = "https://github.com/asaveraasad-data/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL], check=True)

os.chdir(REPO_DIR)

print("Current directory:", os.getcwd())


Current directory: /content/flyrank-ml-internship


## Unit of Analysis

**One row = one content page for one client on one report date (daily performance).**

For this assignment I am using the **fact_content_daily_performance** table from the FlyRank internship warehouse.

## Time Window

I use the **March 2026** partition (2026-03-01 to 2026-03-31).

This month is a mid-panel month recommended by FlyRank because it avoids using the final month for feature development or label creation.

In [14]:
import pandas as pd

print("Rows:", len(df))
print("Start:", df["report_date"].min())
print("End:", df["report_date"].max())

print(df.head())

Rows: 9841378
Start: 2026-03-01 00:00:00
End: 2026-03-31 00:00:00
  report_date           client_hash_id           content_hash_id  \
0  2026-03-01  client_73cda7b4e4f265ea  content_b7e512995f79d5a6   
1  2026-03-01  client_73cda7b4e4f265ea  content_05597932fe4da067   
2  2026-03-01  client_73cda7b4e4f265ea  content_7a105f548d9c6916   
3  2026-03-01  client_73cda7b4e4f265ea  content_905aa32a0230694e   
4  2026-03-01  client_73cda7b4e4f265ea  content_a3ea9792f793ec72   

   client_has_gsc  client_has_ga4  gsc_data_available  ga4_data_available  \
0            True           False                True                <NA>   
1            True           False                True                <NA>   
2            True           False                True                <NA>   
3            True           False                True                <NA>   
4            True           False                True                <NA>   

   gsc_impressions  gsc_clicks  gsc_sum_position  ...  session

## 2. Fields: Feature / Label / Context / Excluded

### Features
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- scroll_events

These features are available before making the content refresh decision.

### Label / Proxy

The future target will be whether a page should be refreshed (or a ranking score). This is not used as an input feature.

### Context

- client_hash_id
- content_hash_id
- report_date

These identify or group rows but are not model features.

### Excluded

- trend_direction
- trend_pct
- is_declining_label

These contain future information or label-derived values and would leak information into the model.

In [15]:
field_contract = {
    "Feature": [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_pageviews",
        "scroll_events",
    ],
    "Context": [
        "client_hash_id",
        "content_hash_id",
        "report_date",
    ],
    "Excluded": [
        "trend_direction",
        "trend_pct",
        "is_declining_label",
    ]
}

for category, cols in field_contract.items():
    print(f"\n{category}")
    for col in cols:
        exists = col in df.columns
        print(f" - {col} {'✓' if exists else '(not in this warehouse table)'}")

print("\nFeature Preview")
df[field_contract["Feature"]].head()


Feature
 - gsc_impressions ✓
 - gsc_clicks ✓
 - gsc_avg_position ✓
 - ga4_pageviews ✓
 - scroll_events ✓

Context
 - client_hash_id ✓
 - content_hash_id ✓
 - report_date ✓

Excluded
 - trend_direction (not in this warehouse table)
 - trend_pct (not in this warehouse table)
 - is_declining_label (not in this warehouse table)

Feature Preview


,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,scroll_events
0,20,0,3.350000,<NA>,<NA>
1,1,0,0.000000,<NA>,<NA>
2,125,1,4.928000,<NA>,<NA>
3,7,0,4.000000,<NA>,<NA>
4,11,0,2.272727,<NA>,<NA>


## 3. Verify it with queries

The following checks verify the data contract.

1. Verify the row count and date range.
2. Verify that rows with Search Console data are available.
3. Verify the dataset grain using the identifier columns.

In [16]:
print("Total rows:", len(df))
print("Date range:")
print(df["report_date"].min(), "to", df["report_date"].max())

print()

available = df[df["gsc_data_available"] == True]

print("Rows with GSC available:", len(available))
print("Percentage:", round(len(available)/len(df)*100,2), "%")

print()

duplicates = (
    df.groupby(["report_date","client_hash_id","content_hash_id"])
      .size()
      .reset_index(name="count")
)

duplicate_rows = duplicates[duplicates["count"] > 1]

print("Duplicate grain rows:", len(duplicate_rows))

Total rows: 9841378
Date range:
2026-03-01 00:00:00 to 2026-03-31 00:00:00

Rows with GSC available: 3611061
Percentage: 36.69 %

Duplicate grain rows: 0


In [17]:
features = available[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_pageviews",
        "scroll_events"
    ]
]

features.head()

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,scroll_events
0,20,0,3.350000,<NA>,<NA>
1,1,0,0.000000,<NA>,<NA>
2,125,1,4.928000,<NA>,<NA>
3,7,0,4.000000,<NA>,<NA>
4,11,0,2.272727,<NA>,<NA>


### Feature availability

**gsc_impressions**

Knowable at the decision moment because Search Console has already recorded impressions before the refresh decision.

**gsc_clicks**

Available before the decision because click counts are historical observations.

**gsc_avg_position**

Represents historical ranking information available before deciding whether to refresh content.

**ga4_pageviews**

Historical Analytics data collected before the decision.

**scroll_events**

Historical engagement data collected before the decision.

### Leakage experiment

The warehouse release intentionally excludes FlyRank's internal label columns.

If a future outcome such as **trend_direction** or **is_declining_label** were added as a feature, model performance would become unrealistically high because the feature directly contains future information.

The experiment was therefore demonstrated conceptually and the column was deliberately excluded from the feature set.

## 4. Data limits

This dataset has several limitations.

- Client history lengths differ across clients.
- Some rows have Search Console data but no GA4 data.
- The warehouse contains historical observations only and cannot explain why traffic changed.
- Only about 36.7% of rows in March 2026 have Search Console data available after filtering.
- This analysis is intended for decision support rather than proving causal relationships.

## Self-check

Before you submit, confirm each line honestly:

✅Every section above is filled — markdown thinking AND the code that backs it

✅The notebook runs top to bottom with no errors (Runtime → Run all)

✅No client names, URLs, or private queries anywhere

✅My claims use careful words: observed, measured, directional, decision-support

✅Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.